# LABORATORIO 2 – ANALIZADOR SINTÁCTICO PARA MINIJAVA

**Asignatura:** Compiladores  
**Estudiante:** Maria Alejandra Arias Medina  
**NRC:** 2439

En este laboratorio desarrollé un analizador sintáctico (parser) para el lenguaje MiniJava definido en el Laboratorio 1, usando Python y la librería Pyparsing. El programa lee un archivo `.txt`, valida su estructura gramatical, construye el árbol de análisis sintáctico como listas anidadas y, cuando encuentra errores, los reporta con su línea, se recupera y continúa analizando el resto del programa.

# Gramática libre de contexto de MiniJava

## 1. Definición

Para establecer qué instrucciones puede reconocer el programa, definí una gramática libre de contexto expresada mediante la cuádrupla:

$$G = (N,\ T,\ P,\ S)$$

- $N$: símbolos no terminales, que representan las estructuras del lenguaje.
- $T$: símbolos terminales, es decir, los tokens del Laboratorio 1 (palabras reservadas, identificadores, literales, operadores y delimitadores).
- $P$: producciones, las reglas que indican cómo se combinan los símbolos.
- $S$: símbolo inicial, $S = \mathit{Programa}$.

Notación: los no terminales van en *cursiva*, los terminales en `monoespaciado`, la barra $\mid$ separa alternativas, $\varepsilon$ es la cadena vacía y $[\ \alpha\ ]$ indica que $\alpha$ es opcional.

## 2. Símbolos no terminales y símbolo inicial

$$
\begin{aligned}
N = \{\ & \mathit{Programa},\ \mathit{ListaSentencias},\ \mathit{Sentencia},\ \mathit{Bloque},\ \mathit{Declaracion},\ \mathit{Tipo},\\
& \mathit{Asignacion},\ \mathit{OpAsignacion},\ \mathit{Incremento},\ \mathit{OpIncremento},\ \mathit{Impresion},\ \mathit{MetodoSalida},\\
& \mathit{Salto},\ \mathit{Condicional},\ \mathit{Alternativa},\ \mathit{CicloWhile},\ \mathit{CicloDoWhile},\ \mathit{CicloFor},\\
& \mathit{InicioFor},\ \mathit{PasoFor},\ \mathit{Seleccion},\ \mathit{ListaCasos},\ \mathit{Caso},\ \mathit{OpcionDefault},\ \mathit{LiteralCaso},\\
& \mathit{Expresion},\ \mathit{ExpOr},\ \mathit{ExpAnd},\ \mathit{ExpRel},\ \mathit{OpRel},\ \mathit{ExpAditiva},\ \mathit{ExpMultiplicativa},\\
& \mathit{ExpUnaria},\ \mathit{Primario},\ \mathit{Literal}\ \}
\end{aligned}
$$

$$S = \mathit{Programa}$$

## 3. Símbolos terminales

Los terminales son los tokens reconocidos por el analizador léxico del Laboratorio 1. Los identificadores y literales se representan por su categoría, porque su lexema cambia de un programa a otro.

| Categoría | Terminales |
|---|---|
| Tipos de datos | `int`, `float`, `boolean`, `bool`, `char`, `String`, `string` |
| Identificadores y literales | `id`, `entero`, `real`, `cadena`, `caracter`, `true`, `false` |
| Control de flujo | `if`, `else`, `while`, `do`, `for`, `switch`, `case`, `default`, `break`, `continue` |
| Salida por consola | `System`, `out`, `print`, `println`, `.` |
| Asignación | `=`, `+=`, `-=`, `*=`, `/=` |
| Aritméticos | `+`, `-`, `*`, `/`, `%`, `++`, `--` |
| Relacionales y lógicos | `<`, `<=`, `>`, `>=`, `==`, `!=`, `&&`, `\|\|`, `!` |
| Delimitadores | `(`, `)`, `{`, `}`, `;`, `:` |

Los espacios, tabulaciones, saltos de línea y comentarios (`//` y `/* */`) no son terminales: se descartan antes del análisis sintáctico.

## 4. Producciones

### 4.1 Programa, bloques y sentencias

$$
\begin{aligned}
\mathit{Programa} &\rightarrow \mathit{ListaSentencias}\\
\mathit{ListaSentencias} &\rightarrow \mathit{Sentencia}\ \mathit{ListaSentencias} \mid \varepsilon\\
\mathit{Sentencia} &\rightarrow \mathit{Declaracion} \mid \mathit{Asignacion} \mid \mathit{Incremento} \mid \mathit{Impresion} \mid \mathit{Salto}\\
&\mid \mathit{Condicional} \mid \mathit{CicloWhile} \mid \mathit{CicloDoWhile} \mid \mathit{CicloFor} \mid \mathit{Seleccion} \mid \mathit{Bloque}\\
\mathit{Bloque} &\rightarrow \texttt{\{}\ \mathit{ListaSentencias}\ \texttt{\}}
\end{aligned}
$$

### 4.2 Sentencias simples

$$
\begin{aligned}
\mathit{Tipo} &\rightarrow \texttt{int} \mid \texttt{float} \mid \texttt{boolean} \mid \texttt{bool} \mid \texttt{char} \mid \texttt{String} \mid \texttt{string}\\
\mathit{Declaracion} &\rightarrow \mathit{Tipo}\ \texttt{id}\ \texttt{;} \mid \mathit{Tipo}\ \texttt{id}\ \texttt{=}\ \mathit{Expresion}\ \texttt{;}\\
\mathit{Asignacion} &\rightarrow \texttt{id}\ \mathit{OpAsignacion}\ \mathit{Expresion}\ \texttt{;}\\
\mathit{OpAsignacion} &\rightarrow \texttt{=} \mid \texttt{+=} \mid \texttt{-=} \mid \texttt{*=} \mid \texttt{/=}\\
\mathit{Incremento} &\rightarrow \texttt{id}\ \mathit{OpIncremento}\ \texttt{;}\\
\mathit{OpIncremento} &\rightarrow \texttt{++} \mid \texttt{-{}-}\\
\mathit{Impresion} &\rightarrow \mathit{MetodoSalida}\ \texttt{(}\ \mathit{Expresion}\ \texttt{)}\ \texttt{;}\\
\mathit{MetodoSalida} &\rightarrow \texttt{System}\ \texttt{.}\ \texttt{out}\ \texttt{.}\ \texttt{print} \mid \texttt{System}\ \texttt{.}\ \texttt{out}\ \texttt{.}\ \texttt{println} \mid \texttt{print}\\
\mathit{Salto} &\rightarrow \texttt{break}\ \texttt{;} \mid \texttt{continue}\ \texttt{;}
\end{aligned}
$$

### 4.3 Condicional (if / else if / else)

$$
\begin{aligned}
\mathit{Condicional} &\rightarrow \texttt{if}\ \texttt{(}\ \mathit{Expresion}\ \texttt{)}\ \mathit{Bloque}\ \mathit{Alternativa}\\
\mathit{Alternativa} &\rightarrow \texttt{else}\ \mathit{Bloque} \mid \texttt{else}\ \mathit{Condicional} \mid \varepsilon
\end{aligned}
$$

### 4.4 Ciclos (while, do-while, for)

$$
\begin{aligned}
\mathit{CicloWhile} &\rightarrow \texttt{while}\ \texttt{(}\ \mathit{Expresion}\ \texttt{)}\ \mathit{Bloque}\\
\mathit{CicloDoWhile} &\rightarrow \texttt{do}\ \mathit{Bloque}\ \texttt{while}\ \texttt{(}\ \mathit{Expresion}\ \texttt{)}\ \texttt{;}\\
\mathit{CicloFor} &\rightarrow \texttt{for}\ \texttt{(}\ \mathit{InicioFor}\ \texttt{;}\ \mathit{Expresion}\ \texttt{;}\ \mathit{PasoFor}\ \texttt{)}\ \mathit{Bloque}\\
\mathit{InicioFor} &\rightarrow \mathit{Tipo}\ \texttt{id}\ \texttt{=}\ \mathit{Expresion} \mid \texttt{id}\ \texttt{=}\ \mathit{Expresion}\\
\mathit{PasoFor} &\rightarrow \texttt{id}\ \mathit{OpIncremento} \mid \texttt{id}\ \mathit{OpAsignacion}\ \mathit{Expresion} \mid \varepsilon
\end{aligned}
$$

Si $\mathit{PasoFor} \rightarrow \varepsilon$, el paso vale $1$ cuando la condición es creciente (`<`, `<=`) y $-1$ cuando es decreciente (`>`, `>=`). La declaración múltiple con comas (`int i = 0, n = 5`) no pertenece a $\mathit{InicioFor}$ y se reporta como error de encabezado.

### 4.5 Selección múltiple (switch-case)

$$
\begin{aligned}
\mathit{Seleccion} &\rightarrow \texttt{switch}\ \texttt{(}\ \mathit{Expresion}\ \texttt{)}\ \texttt{\{}\ \mathit{ListaCasos}\ \mathit{OpcionDefault}\ \texttt{\}}\\
\mathit{ListaCasos} &\rightarrow \mathit{Caso}\ \mathit{ListaCasos} \mid \varepsilon\\
\mathit{Caso} &\rightarrow \texttt{case}\ \mathit{LiteralCaso}\ \texttt{:}\ \mathit{ListaSentencias}\\
\mathit{OpcionDefault} &\rightarrow \texttt{default}\ \texttt{:}\ \mathit{ListaSentencias} \mid \varepsilon\\
\mathit{LiteralCaso} &\rightarrow \mathit{Literal} \mid \texttt{-}\ \texttt{entero} \mid \texttt{-}\ \texttt{real}
\end{aligned}
$$

### 4.6 Expresiones (jerarquía de precedencia)

Cada nivel de precedencia es un no terminal distinto: cuanto más abajo en la jerarquía, mayor precedencia. La recursión por la izquierda en los operadores binarios expresa la asociatividad izquierda y la recursión por la derecha en los unarios expresa la asociatividad derecha.

| Nivel | Operadores | Asociatividad | No terminal |
|---|---|---|---|
| 1 (mayor) | `!`  `-` (unarios) | derecha | $\mathit{ExpUnaria}$ |
| 2 | `*`  `/`  `%` | izquierda | $\mathit{ExpMultiplicativa}$ |
| 3 | `+`  `-` | izquierda | $\mathit{ExpAditiva}$ |
| 4 | `<`  `<=`  `>`  `>=`  `==`  `!=` | izquierda | $\mathit{ExpRel}$ |
| 5 | `&&` | izquierda | $\mathit{ExpAnd}$ |
| 6 (menor) | `\|\|` | izquierda | $\mathit{ExpOr}$ |

$$
\begin{aligned}
\mathit{Expresion} &\rightarrow \mathit{ExpOr}\\
\mathit{ExpOr} &\rightarrow \mathit{ExpOr}\ \texttt{||}\ \mathit{ExpAnd} \mid \mathit{ExpAnd}\\
\mathit{ExpAnd} &\rightarrow \mathit{ExpAnd}\ \texttt{\&\&}\ \mathit{ExpRel} \mid \mathit{ExpRel}\\
\mathit{ExpRel} &\rightarrow \mathit{ExpRel}\ \mathit{OpRel}\ \mathit{ExpAditiva} \mid \mathit{ExpAditiva}\\
\mathit{OpRel} &\rightarrow \texttt{<} \mid \texttt{<=} \mid \texttt{>} \mid \texttt{>=} \mid \texttt{==} \mid \texttt{!=}\\
\mathit{ExpAditiva} &\rightarrow \mathit{ExpAditiva}\ \texttt{+}\ \mathit{ExpMultiplicativa} \mid \mathit{ExpAditiva}\ \texttt{-}\ \mathit{ExpMultiplicativa} \mid \mathit{ExpMultiplicativa}\\
\mathit{ExpMultiplicativa} &\rightarrow \mathit{ExpMultiplicativa}\ \texttt{*}\ \mathit{ExpUnaria} \mid \mathit{ExpMultiplicativa}\ \texttt{/}\ \mathit{ExpUnaria} \mid \mathit{ExpMultiplicativa}\ \texttt{\%}\ \mathit{ExpUnaria} \mid \mathit{ExpUnaria}\\
\mathit{ExpUnaria} &\rightarrow \texttt{!}\ \mathit{ExpUnaria} \mid \texttt{-}\ \mathit{ExpUnaria} \mid \mathit{Primario}\\
\mathit{Primario} &\rightarrow \texttt{id} \mid \mathit{Literal} \mid \texttt{(}\ \mathit{Expresion}\ \texttt{)}\\
\mathit{Literal} &\rightarrow \texttt{entero} \mid \texttt{real} \mid \texttt{cadena} \mid \texttt{caracter} \mid \texttt{true} \mid \texttt{false}
\end{aligned}
$$

Así, `4 + 2 * 3` se agrupa como una suma cuyo segundo operando es la multiplicación. En el código esta jerarquía no se escribe a mano: se construye con `infix_notation` de Pyparsing (que además elimina la recursión por la izquierda), pasando los niveles en el mismo orden de la tabla.

## 5. Resolución del *dangling else*

La ambigüedad del else colgante aparece en gramáticas del tipo $S \rightarrow \texttt{if}\ (E)\ S \mid \texttt{if}\ (E)\ S\ \texttt{else}\ S$, donde en `if (a) if (b) x; else y;` no se sabe a cuál `if` pertenece el `else`. En esta gramática se resuelve así:

1. **Por construcción:** el cuerpo de un `if` y de un `else` es siempre un $\mathit{Bloque}$ delimitado por `{` y `}`. Las llaves marcan dónde termina cada `if`, de modo que un `else` solo puede aparecer justo después de la llave de cierre del `if` al que pertenece. Por ejemplo, en `if (a) { if (b) {...} else {...} }` el `else` pertenece al `if` interno, y en `if (a) { if (b) {...} } else {...}` pertenece al externo.
2. **Por convención:** en las cadenas `else if`, el `else` se asocia con el `if` pendiente más cercano. En el código, la alternativa es `Optional(else ...)`, que es voraz: el `if` más interno toma el `else` que le sigue. Un `else if` se representa en el árbol como un bloque `else` que contiene un único `if`.

## 6. Producciones de error

Para que el análisis no se detenga ante el primer error, la gramática se amplía con producciones de error (el símbolo $\mathbf{error}$ representa el nodo `['error', contexto, descripcion]`):

| Producción de error | Contexto – descripción |
|---|---|
| $\mathit{Declaracion} \rightarrow \mathit{Tipo}\ \texttt{id}\ [\texttt{=}\ \mathit{Expresion}]\ \mathbf{error}$ | `declaracion` – falta punto y coma |
| $\mathit{Asignacion} \rightarrow \texttt{id}\ \mathit{OpAsignacion}\ \mathit{Expresion}\ \mathbf{error}$ | `asignacion` – falta punto y coma |
| $\mathit{Incremento} \rightarrow \texttt{id}\ \mathit{OpIncremento}\ \mathbf{error}$ | `incremento` – falta punto y coma |
| $\mathit{Salto} \rightarrow \texttt{break}\ \mathbf{error} \mid \texttt{continue}\ \mathbf{error}$ | `break` / `continue` – falta punto y coma |
| $\mathit{Impresion} \rightarrow \mathit{MetodoSalida}\ \texttt{(}\ \mathit{Expresion}\ \texttt{)}\ \mathbf{error}$ | `print` – falta punto y coma |
| $\mathit{Impresion} \rightarrow \mathit{MetodoSalida}\ \texttt{(}\ \mathit{Expresion}\ \mathbf{error}$ | `print` – falta parentesis de cierre |
| Condición $\rightarrow \texttt{(}\ \texttt{)}$ | `condicion` – condicion vacia |
| Condición $\rightarrow \texttt{(}\ \mathbf{error}\ \texttt{)}$ | `condicion` – expresion incompleta |
| Condición $\rightarrow \texttt{(}\ \mathit{Expresion}\ \mathbf{error}$ | `if` / `while` / `dowhile` / `switch` – falta parentesis de cierre |
| $\mathit{CicloDoWhile} \rightarrow \texttt{do}\ \mathit{Bloque}\ \texttt{while}\ \texttt{(}\ \mathit{Expresion}\ \texttt{)}\ \mathbf{error}$ | `dowhile` – falta punto y coma |
| $\mathit{CicloFor} \rightarrow \texttt{for}\ \mathbf{error}\ \mathit{Bloque}$ | `for` – faltan separadores ';' en el encabezado (o encabezado mal formado) |
| $\mathit{Caso} \rightarrow \texttt{case}\ \mathit{LiteralCaso}\ \mathbf{error}\ \mathit{ListaSentencias}$ | `case` – faltan dos puntos |
| $\mathit{Bloque} \rightarrow \texttt{\{}\ \mathit{ListaSentencias}\ \mathbf{error}$ | `bloque` – falta llave de cierre |
| $\mathit{Sentencia} \rightarrow \mathbf{error}\ \texttt{;}$ | `sentencia` – sentencia no reconocida |

La última es la regla de último recurso: descarta el texto hasta el siguiente punto y coma o llave (sin pasar de la línea actual) y el análisis continúa con la sentencia siguiente.

# Manual de uso del analizador

## 1. Descripción

Este notebook revisa la sintaxis de un programa MiniJava guardado en un archivo de texto `.txt`. El analizador está construido en Python con la librería **Pyparsing** y:

- valida la estructura gramatical del programa;
- construye el árbol de análisis sintáctico como listas anidadas;
- reporta cada error sintáctico con su línea, se recupera y sigue analizando el resto del archivo.

El programa no ejecuta el código MiniJava: solo valida su forma. Tampoco revisa errores léxicos ni semánticos (tipos, variables no declaradas, etc.).

## 2. Requisitos y librerías

- Google Colab, o Python 3.8+ con Jupyter Notebook.
- **Pyparsing** (versión 3.x): construye la gramática (`Forward`, `Group`, `MatchFirst`, `Optional`, `ZeroOrMore`, `OneOrMore`, `Suppress`, `Keyword`, `Literal`, `Regex`, `infix_notation`). En Google Colab ya viene instalada. Si no está disponible:
  - En una celda de Colab o Jupyter: `!pip install pyparsing`
  - En una terminal: `python -m pip install pyparsing`
- **re** (incluida en Python): se usa solo para eliminar los comentarios antes del análisis.

No se usa ningún generador de analizadores (PLY, ANTLR, Lark, lex/yacc).

## 3. Estructura del notebook

1. **Celda 1:** título, autora y NRC.
2. **Celda 2:** gramática libre de contexto (no terminales, terminales, producciones, precedencia, dangling else y producciones de error).
3. **Celda 3:** este manual.
4. **Celda 4:** código del analizador sintáctico.

## 4. Pasos para ejecutar

1. Abrir el archivo `.ipynb` en Google Colab (o en Jupyter).
2. Ejecutar la **celda 4** (o *Entorno de ejecución → Ejecutar todas*).
3. **En Colab** aparece el botón **"Elegir archivos"**: seleccionar el archivo `.txt` que se quiere analizar.
   **En Jupyter o Python local** el programa pide la ruta del archivo; por ejemplo: `C:\pruebas\programa.txt`.
4. El resultado aparece debajo de la celda.
5. Para analizar otro archivo, volver a ejecutar la celda 4.

El archivo puede tener cualquier nombre, pero debe tener extensión `.txt` y estar en texto plano (UTF-8 preferiblemente).

## 5. Formato del archivo de entrada

El programa es una secuencia de sentencias MiniJava (no hace falta definir clases ni métodos). Se admiten declaraciones, asignaciones (`=`, `+=`, `-=`, `*=`, `/=`), `++` y `--`, `System.out.print` / `System.out.println`, `if`/`else`/`else if`, `while`, `do-while`, `for`, `switch`/`case`/`default`, `break`, `continue`, bloques `{ }` anidados y comentarios `//` y `/* */`. Por ejemplo:

```java
int contador = 0;
// comentario
while (contador < 3) {
    contador++;
    System.out.println(contador);
}
```

## 6. Interpretación de la salida

**Programa correcto:** se imprime `Codigo Valido` y luego el árbol:

```
Codigo Valido

Arbol de Analisis Sintactico

[
  ['int', 'contador', '0'],
  [
    'while',
    ['contador', '<', '3'],
    [
      ['contador', '++'],
      ['println', 'contador']
    ]
  ]
]
```

**Programa con errores:** se imprime `Errores de sintaxis`, la lista de errores ordenada por línea con el formato `ERROR SINTACTICO (linea N): contexto - descripcion` y, a continuación, el árbol con los nodos `['error', contexto, descripcion]` en el lugar del elemento que no se pudo reconocer.

Reglas del árbol:

- El programa y cada bloque son la lista de sus sentencias.
- Una expresión atómica es su lexema (`'x'`, `'10'`, `'"Hola"'`); una compuesta es la lista de operandos y operadores en notación infija, y cada nivel de precedencia distinto se anida.
- En declaraciones, asignaciones e impresiones la expresión se funde con el nodo: `z = z + i;` → `['z', 'z', '+', 'i']`.
- Estructuras: `['if', cond, bloque, bloqueElse]`, `['while', cond, bloque]`, `['dowhile', bloque, cond]`, `['for', [tipo, var, inicial, cond, paso], bloque]`, `['switch', expr, [casos]]`, `['case', literal, bloque]`, `['default', bloque]`.
- Si el `for` no tiene paso, se coloca `[var, '1']` (condición `<`, `<=`) o `[var, '-1']` (condición `>`, `>=`).

## 7. Errores que se detectan

| Contexto | Descripción | Ejemplo |
|---|---|---|
| declaracion | falta punto y coma | `int x = 10` |
| asignacion | falta punto y coma | `x = 1` |
| incremento | falta punto y coma | `contador++` |
| break / continue | falta punto y coma | `break` |
| print | falta punto y coma | `System.out.print(x)` |
| print | falta parentesis de cierre | `System.out.print(x;` |
| condicion | condicion vacia | `if () {` |
| condicion | expresion incompleta | `if (x > ) {` |
| if / while / dowhile / switch | falta parentesis de cierre | `while (x < 20 {` |
| dowhile | falta punto y coma | `} while (x > 0)` |
| for | faltan separadores ';' en el encabezado | `for (int i=0; i<3 i++)` |
| for | encabezado mal formado | `for (int i = 0, n = 5; i < n; i++)` |
| case | faltan dos puntos | `case 1` |
| bloque | falta llave de cierre | `if (x) { y = 1;` |
| sentencia | sentencia no reconocida | `int 456 = 7;` |

Los errores de punto y coma se reportan en la línea donde empieza la sentencia, y el de llave faltante en la línea donde se abrió el bloque.

## 8. Archivos de prueba incluidos

- `1 programa.txt` y `2 programa.txt`: programas válidos del enunciado.
- `3 programa.txt` y `4 programa.txt`: programas con errores del enunciado.
- `Prueba validos.txt` y `Prueba con errores.txt`: pruebas adicionales.

La salida de los cuatro programas del enunciado coincide exactamente con la salida esperada de la sección 8 del enunciado.

## 9. Limitaciones

- Reconoce MiniJava tal como se definió en el laboratorio, no todo Java (no hay clases, métodos, arreglos ni funciones de usuario).
- Las sentencias `if`, `while`, `for` y `do` deben usar bloques con llaves.
- Como forma abreviada, también se acepta `print(x);`.
- Cuando una misma línea tiene varios errores, se reporta el primero que impide reconocer la sentencia.

In [ ]:
# =====================================================================
#  ANALIZADOR SINTACTICO PARA MINIJAVA (Pyparsing)
# =====================================================================
import re
import pyparsing as pp
from pyparsing import (Forward, Group, MatchFirst, Optional, ZeroOrMore, OneOrMore,
                       Suppress, Keyword, Literal, Regex, infix_notation, OpAssoc,
                       StringEnd)

pp.ParserElement.enable_packrat()


# ---------------------------------------------------------------------
# 1. Nodos auxiliares del arbol
# ---------------------------------------------------------------------
class Nodo:
    """Envuelve una lista para que Pyparsing no la aplane al devolverla
    desde una accion de analisis."""
    def __init__(self, valor):
        self.valor = valor

    def __repr__(self):
        return repr(self.valor)


class ErrorSintactico:
    """Nodo de error ['error', contexto, descripcion] que recuerda su linea.
    Los errores se recogen al final recorriendo el arbol; asi solo se
    reportan los que quedaron en el analisis definitivo y no los de
    alternativas que Pyparsing descarto al retroceder."""
    def __init__(self, linea, contexto, descripcion):
        self.linea = linea
        self.contexto = contexto
        self.descripcion = descripcion

    def __repr__(self):
        return repr(["error", self.contexto, self.descripcion])


def a_lista(x):
    """Convierte ParseResults / Nodo en listas de Python."""
    if isinstance(x, Nodo):
        return a_lista(x.valor)
    if isinstance(x, (pp.ParseResults, list)):
        return [a_lista(v) for v in x]
    return x


def fundir(prefijo, expresion):
    """Regla 3.3: si la expresion es atomica se agrega como un elemento;
    si es compuesta se agregan sus elementos."""
    e = a_lista(expresion)
    return prefijo + (e if isinstance(e, list) else [e])


def buscar_error(tokens):
    for t in tokens:
        if isinstance(t, ErrorSintactico):
            return t
    return None


def en_linea(error, s, l):
    """Copia del error ubicada en la linea donde empieza la sentencia."""
    return ErrorSintactico(pp.lineno(l, s), error.contexto, error.descripcion)


def accion_sentencia(constructor):
    """Accion de analisis de una sentencia simple: si alguna produccion de
    error se activo, el nodo de error reemplaza a toda la sentencia y se
    reporta en la linea donde la sentencia comienza."""
    def accion(s, l, t):
        error = buscar_error(t)
        if error:
            return en_linea(error, s, l)
        return Nodo(constructor(t))
    return accion


# ---------------------------------------------------------------------
# 2. Herramientas para las producciones de error
# ---------------------------------------------------------------------
def produccion_error(expresion, contexto, descripcion):
    """Convierte una expresion en una produccion de error: si reconoce el
    texto, devuelve el nodo ['error', contexto, descripcion]."""
    return expresion.set_parse_action(
        lambda s, l, t: ErrorSintactico(pp.lineno(l, s), contexto, descripcion))


def ultimo_token(s, l):
    """Posicion del ultimo caracter no blanco antes de `l`."""
    j = l - 1
    while j > 0 and s[j] in " \t\n":
        j -= 1
    return max(j, 0)


def falta_elemento(condicion_fn, contexto, descripcion):
    """Produccion de error que no consume texto (un simbolo que falta).
    Se acepta solo si condicion_fn(texto, posicion) es verdadera, y el error
    se ubica en la linea del ultimo token reconocido."""
    return pp.Empty().add_condition(lambda s, l, t: condicion_fn(s, l)).add_parse_action(
        lambda s, l, t: ErrorSintactico(pp.lineno(ultimo_token(s, l), s), contexto, descripcion))


def termina_linea(s, l):
    """Hay un salto de linea entre el ultimo token y la posicion actual,
    o lo que sigue es '}' o el fin del archivo."""
    return l >= len(s) or s[l] == "}" or "\n" in s[ultimo_token(s, l) + 1:l]


def sigue(caracter):
    return lambda s, l: s.startswith(caracter, l)


def pyc_o_fin_linea(s, l):
    return s.startswith(";", l) or termina_linea(s, l)


def fin_texto(s, l):
    return l >= len(s)


def siempre(s, l):
    return True


# ---------------------------------------------------------------------
# 3. Terminales (tokens del Laboratorio 1)
# ---------------------------------------------------------------------
TIPOS = ("int", "float", "boolean", "bool", "char", "String", "string")
RESERVADAS = set(TIPOS) | {"if", "else", "while", "do", "for", "switch", "case",
                           "default", "break", "continue", "true", "false"}

LP, RP = Suppress("("), Suppress(")")
LB, RB = Suppress("{"), Suppress("}")
SEMI, COLON = Suppress(";"), Suppress(":")

identificador = Regex(r"[A-Za-z_][A-Za-z0-9_]*").add_condition(
    lambda t: t[0] not in RESERVADAS)
PATRON_NUMERO = r"\d+\.\d+(?:[eE][+-]?\d+)?|\d+[eE][+-]?\d+|\d+"
numero = Regex(PATRON_NUMERO)
cadena = Regex(r'"(?:[^"\\\n]|\\.)*"')
caracter = Regex(r"'(?:[^'\\\n]|\\.)'")
booleano = MatchFirst([Keyword("true"), Keyword("false")])
tipo = MatchFirst([Keyword(t) for t in TIPOS])
op_asignacion = MatchFirst([Literal(o) for o in ("+=", "-=", "*=", "/=", "=")])
op_incremento = MatchFirst([Literal("++"), Literal("--")])

# ---------------------------------------------------------------------
# 4. Expresiones: infix_notation resuelve precedencia y asociatividad
#    (de mayor a menor: unarios, * / %, + -, relacionales, &&, ||)
# ---------------------------------------------------------------------
operando = MatchFirst([numero, cadena, caracter, booleano, identificador])
expresion = infix_notation(operando, [
    (MatchFirst([Literal("!"), Literal("-")]), 1, OpAssoc.RIGHT),
    (MatchFirst([Literal("*"), Literal("/"), Literal("%")]), 2, OpAssoc.LEFT),
    (MatchFirst([Literal("+"), Literal("-")]), 2, OpAssoc.LEFT),
    (MatchFirst([Literal(o) for o in ("<=", ">=", "==", "!=", "<", ">")]), 2, OpAssoc.LEFT),
    (Literal("&&"), 2, OpAssoc.LEFT),
    (Literal("||"), 2, OpAssoc.LEFT),
])


def fin_sentencia(contexto):
    """';' obligatorio. Si falta y la linea ya termino, produce el error."""
    return SEMI | falta_elemento(termina_linea, contexto, "falta punto y coma")


def condicion(contexto, siguiente):
    """'(' Expresion ')' con sus producciones de error.
    `siguiente` indica lo que debe venir despues del parentesis
    ('{' en if/while/switch; ';' o fin de linea en do-while)."""
    return MatchFirst([
        produccion_error(LP + RP, "condicion", "condicion vacia"),
        LP + expresion + RP,
        (LP + expresion + falta_elemento(siguiente, contexto, "falta parentesis de cierre")
         ).set_parse_action(lambda t: t[-1]),
        produccion_error(Regex(r"\([^{};]*\)"), "condicion", "expresion incompleta"),
        produccion_error(Regex(r"\([^{};)]*"), contexto, "falta parentesis de cierre"),
    ])


# ---------------------------------------------------------------------
# 5. Sentencias (Forward: una sentencia contiene bloques y viceversa)
# ---------------------------------------------------------------------
sentencia = Forward()
bloque = Forward()
condicional = Forward()
lista_sentencias = Optional(OneOrMore(sentencia))

# 5.1 Declaracion:  Tipo ID [= Expresion] ;
declaracion = tipo + identificador + Optional(Suppress("=") + expresion) \
    + fin_sentencia("declaracion")


declaracion.set_parse_action(accion_sentencia(
    lambda t: fundir([t[0], t[1]], t[2]) if len(t) == 3 else [t[0], t[1]]))

# 5.2 Asignacion:  ID (= | += | -= | *= | /=) Expresion ;
#     En '=' el operador no aparece en el arbol: z = 0;  ->  ['z', '0']
asignacion = identificador + op_asignacion + expresion + fin_sentencia("asignacion")
asignacion.set_parse_action(accion_sentencia(
    lambda t: fundir([t[0]] if t[1] == "=" else [t[0], t[1]], t[2])))

# 5.3 Incremento / decremento:  ID (++ | --) ;
incremento = identificador + op_incremento + fin_sentencia("incremento")
incremento.set_parse_action(accion_sentencia(lambda t: [t[0], t[1]]))

# 5.4 Salida por consola:  System.out.(print | println) ( Expresion ) ;
metodo_salida = MatchFirst([
    Suppress(Keyword("System")) + Suppress(".") + Suppress(Keyword("out")) + Suppress(".")
    + MatchFirst([Keyword("println"), Keyword("print")]),
    Keyword("print"),          # forma abreviada print(x); admitida por compatibilidad
])
impresion = metodo_salida + LP + expresion + MatchFirst([
    RP + fin_sentencia("print"),
    falta_elemento(pyc_o_fin_linea, "print", "falta parentesis de cierre") + Optional(SEMI),
])
impresion.set_parse_action(accion_sentencia(lambda t: fundir([t[0]], t[1])))

# 5.5 break ; | continue ;
sentencia_break = Suppress(Keyword("break")) + fin_sentencia("break")
sentencia_break.set_parse_action(accion_sentencia(lambda t: ["break"]))
sentencia_continue = Suppress(Keyword("continue")) + fin_sentencia("continue")
sentencia_continue.set_parse_action(accion_sentencia(lambda t: ["continue"]))

# 5.6 Bloque:  { ListaSentencias }
falta_llave = falta_elemento(fin_texto, "bloque", "falta llave de cierre")
bloque <<= LB + Group(lista_sentencias) + (RB | falta_llave)


def pa_bloque(s, l, t):
    cuerpo = a_lista(t[0])
    if len(t) > 1:     # llave faltante: el error va al final del bloque y se
        cuerpo.append(en_linea(t[1], s, l))   # reporta en la linea del '{'
    return Nodo(cuerpo)


bloque.set_parse_action(pa_bloque)

# 5.7 Condicional:  if ( Expresion ) Bloque [else (Bloque | Condicional)]
#     Dangling else: Optional es voraz, asi que el if mas interno toma
#     el else que le sigue (el else se asocia al if mas cercano).
condicional <<= Suppress(Keyword("if")) + condicion("if", sigue("{")) + bloque \
    + Optional(Suppress(Keyword("else")) + MatchFirst([condicional, bloque]))


def pa_condicional(t):
    nodo = ["if", a_lista(t[0]), a_lista(t[1])]
    if len(t) > 2:
        alternativa = a_lista(t[2])
        if alternativa and alternativa[0] == "if":   # else if -> bloque con un if
            alternativa = [alternativa]
        nodo.append(alternativa)
    return Nodo(nodo)


condicional.set_parse_action(pa_condicional)

# 5.8 while ( Expresion ) Bloque
ciclo_while = Suppress(Keyword("while")) + condicion("while", sigue("{")) + bloque
ciclo_while.set_parse_action(lambda t: Nodo(["while", a_lista(t[0]), a_lista(t[1])]))

# 5.9 do Bloque while ( Expresion ) ;
ciclo_do = Suppress(Keyword("do")) + bloque + Suppress(Keyword("while")) \
    + condicion("dowhile", pyc_o_fin_linea) + fin_sentencia("dowhile")


def pa_do(s, l, t):
    if len(t) > 2 and isinstance(t[2], ErrorSintactico):   # falta ';' final:
        return en_linea(t[2], s, l)                        # el error reemplaza al do-while
    return Nodo(["dowhile", a_lista(t[0]), a_lista(t[1])])


ciclo_do.set_parse_action(pa_do)

# 5.10 for ( Inicio ; Expresion ; [Paso] ) Bloque
inicio_for = Group(MatchFirst([tipo + identificador, identificador])
                   + Suppress("=") + expresion)
paso_for = Group(MatchFirst([identificador + op_incremento,
                             identificador + op_asignacion + expresion]))
ciclo_for = Suppress(Keyword("for")) + LP + inicio_for + SEMI + expresion + SEMI \
    + Optional(paso_for) + RP + bloque


def pa_for(t):
    inicio = a_lista(t[0])
    condicion_for = a_lista(t[1])
    variable = inicio[-2]
    if len(t) == 4:                                   # paso explicito
        p = a_lista(t[2])
        if p[1] in ("++", "--"):
            paso = [p[0], p[1]]
        elif p[1] == "=":
            paso = fundir([p[0]], p[2])
        else:
            paso = fundir([p[0], p[1]], p[2])
    else:                                             # paso omitido: 1 o -1
        operadores = condicion_for if isinstance(condicion_for, list) else []
        decreciente = any(op in (">", ">=") for op in operadores if isinstance(op, str))
        paso = [variable, "-1" if decreciente else "1"]
    encabezado = inicio + [condicion_for, paso]
    return Nodo(["for", encabezado, a_lista(t[-1])])


ciclo_for.set_parse_action(pa_for)

# Produccion de error del for: encabezado que no cumple la forma anterior
encabezado_for_malo = Regex(r"\([^{}]*")


def pa_for_malo(s, l, t):
    texto = t[0].strip()
    if texto.count(";") != 2:
        descripcion = "faltan separadores ';' en el encabezado"
    elif not texto.endswith(")"):
        descripcion = "falta parentesis de cierre"
    else:
        descripcion = "encabezado mal formado"
    return ErrorSintactico(pp.lineno(l, s), "for", descripcion)


encabezado_for_malo.set_parse_action(pa_for_malo)
ciclo_for_error = Suppress(Keyword("for")) + encabezado_for_malo + bloque
ciclo_for_error.set_parse_action(lambda t: Nodo(["for", t[0], a_lista(t[1])]))

# 5.11 switch ( Expresion ) { Caso* [Default] }
literal_caso = MatchFirst([Regex(r"-?(?:" + PATRON_NUMERO + ")"), cadena, caracter, booleano])
fin_de_caso = MatchFirst([Keyword("case"), Keyword("default"), Literal("}")])
cuerpo_caso = Group(ZeroOrMore(~fin_de_caso + sentencia))
caso = Suppress(Keyword("case")) + MatchFirst([
    literal_caso + COLON,
    produccion_error(literal_caso.copy(), "case", "faltan dos puntos"),
    produccion_error(Regex(r"[^:;{}\n]*:"), "case", "literal invalido"),
]) + cuerpo_caso
caso.set_parse_action(lambda t: Nodo(["case", t[0], a_lista(t[1])]))

opcion_default = Suppress(Keyword("default")) \
    + (COLON | falta_elemento(siempre, "default", "faltan dos puntos")) + cuerpo_caso


def pa_default(t):
    if len(t) == 2:
        return Nodo(["default", t[0], a_lista(t[1])])
    return Nodo(["default", a_lista(t[0])])


opcion_default.set_parse_action(pa_default)

seleccion = Suppress(Keyword("switch")) + condicion("switch", sigue("{")) + LB \
    + Group(ZeroOrMore(caso)) + Optional(opcion_default) + (RB | falta_llave)


def pa_switch(t):
    casos = a_lista(t[1]) + [a_lista(x) for x in t[2:]]
    return Nodo(["switch", a_lista(t[0]), casos])


seleccion.set_parse_action(pa_switch)

# 5.12 Regla de ultimo recurso: descarta texto hasta el siguiente ';'
#      (incluido) o hasta antes de una llave, sin pasar de la linea actual
#      para no ocultar las sentencias de las lineas siguientes.
sentencia_no_reconocida = produccion_error(Regex(r"[^;{}\n]+;?|;"),
                                           "sentencia", "sentencia no reconocida")

sentencia <<= MatchFirst([
    declaracion, impresion, incremento, asignacion,
    sentencia_break, sentencia_continue,
    condicional, ciclo_while, ciclo_do, ciclo_for, ciclo_for_error, seleccion,
    bloque, sentencia_no_reconocida,
])

# 5.13 Programa:  ListaSentencias  (una '}' sin '{' tambien se reporta)
llave_sobrante = produccion_error(Literal("}"), "bloque", "llave de cierre sin apertura")
programa = Group(ZeroOrMore(MatchFirst([sentencia, llave_sobrante]))) + StringEnd()


# ---------------------------------------------------------------------
# 6. Preprocesamiento, recoleccion de errores e impresion del arbol
# ---------------------------------------------------------------------
COMENTARIO_O_LITERAL = re.compile(
    r'("(?:[^"\\\n]|\\.)*"|\'(?:[^\'\\\n]|\\.)*\')|(//[^\n]*|/\*[\s\S]*?\*/)')


def quitar_comentarios(codigo):
    """Reemplaza los comentarios por espacios conservando los saltos de
    linea, para que los numeros de linea de los errores no cambien.
    Un "//" dentro de una cadena no se toma como comentario."""
    def reemplazo(m):
        if m.group(1):
            return m.group(1)
        return re.sub(r"[^\n]", " ", m.group(2))
    return COMENTARIO_O_LITERAL.sub(reemplazo, codigo)


def recolectar(nodo, errores):
    """Convierte el arbol a listas simples y junta los errores en orden."""
    if isinstance(nodo, ErrorSintactico):
        errores.append((nodo.linea, nodo.contexto, nodo.descripcion))
        return ["error", nodo.contexto, nodo.descripcion]
    if isinstance(nodo, list):
        return [recolectar(x, errores) for x in nodo]
    return nodo


def formatear(nodo, nivel=0):
    """Arbol con sangria: una lista que solo contiene elementos atomicos va
    en una linea; las demas abren un nivel de sangria por elemento."""
    sangria = "  " * nivel
    if not isinstance(nodo, list):
        return sangria + repr(nodo)
    if all(not isinstance(x, list) for x in nodo):
        return sangria + "[" + ", ".join(repr(x) for x in nodo) + "]"
    partes = [formatear(x, nivel + 1) for x in nodo]
    return sangria + "[\n" + ",\n".join(partes) + "\n" + sangria + "]"


def analizar_codigo(codigo, mostrar=True):
    codigo = codigo.lstrip("﻿").replace("\r\n", "\n").replace("\r", "\n")
    codigo = quitar_comentarios(codigo)
    errores = []
    try:
        resultado = programa.parse_string(codigo, parse_all=True)
        arbol = recolectar(a_lista(resultado[0]), errores)
    except pp.ParseBaseException as exc:      # no deberia ocurrir: hay regla de ultimo recurso
        errores.append((exc.lineno, "sentencia", "error sintactico no recuperado"))
        arbol = []
    errores.sort(key=lambda e: e[0])           # orden por linea (estable)
    if mostrar:
        if errores:
            print("Errores de sintaxis\n")
            for linea, contexto, descripcion in errores:
                print(f"ERROR SINTACTICO (linea {linea}): {contexto} - {descripcion}")
        else:
            print("Codigo Valido")
        print("\nArbol de Analisis Sintactico\n")
        print(formatear(arbol))
    return arbol, errores


# ---------------------------------------------------------------------
# 7. Carga interactiva del archivo .txt
# ---------------------------------------------------------------------
def cargar_archivo():
    try:
        from google.colab import files          # Google Colab
    except ImportError:
        files = None
    if files is not None:
        subidos = files.upload()
        if not subidos:
            raise ValueError("No se selecciono ningun archivo.")
        nombre = next(iter(subidos))
        contenido = subidos[nombre]
    else:                                        # Jupyter / Python local
        nombre = input("Ingrese la ruta del archivo .txt: ").strip().strip('"')
        if not nombre.lower().endswith(".txt"):
            raise ValueError("El archivo debe tener extension .txt.")
        with open(nombre, "rb") as archivo:
            contenido = archivo.read()
    if not nombre.lower().endswith(".txt"):
        raise ValueError("El archivo debe tener extension .txt.")
    try:
        return nombre, contenido.decode("utf-8-sig")
    except UnicodeDecodeError:
        return nombre, contenido.decode("latin-1")


if __name__ == "__main__":
    try:
        nombre_archivo, codigo_fuente = cargar_archivo()
        print(f"Archivo analizado: {nombre_archivo}\n")
        analizar_codigo(codigo_fuente)
    except Exception as error:
        print("No fue posible analizar el archivo:", error)